# 02 — เสถียรภาพของฟีเจอร์ (Feature Stability)

ตรวจสอบว่าฟีเจอร์ Phase 2 ไม่พังที่ขอบ rollover และไม่มี look-ahead leakage

**หมายเหตุ (data-gated):** การรีวิว 5 ปีเต็มต้องใช้ `TVKIT_AUTH_TOKEN` จริงเพื่อดึงข้อมูลย้อนหลัง
(เหมือน backfill ของ Phase 1) โน้ตบุคนี้รันบนข้อมูลที่มีอยู่ใน `data/features/` ถ้ายังไม่มีให้รัน
`uv run python scripts/build_features.py` ก่อน

In [ ]:
from __future__ import annotations

import polars as pl

from tfex_s50_multi_tf_swing.config.settings import get_settings
from tfex_s50_multi_tf_swing.features.models import FeatureConfig
from tfex_s50_multi_tf_swing.features.store import FeatureStore

settings = get_settings()
store = FeatureStore(settings.data_dir, FeatureConfig())

TIMEFRAME = "5m"
try:
    panel = store.read_panel(TIMEFRAME)
    print(f"loaded {TIMEFRAME} panel: {panel.shape}")
except Exception as exc:  # noqa: BLE001 - notebook guard
    panel = None
    print(f"no feature panel yet ({exc}); run scripts/build_features.py first")

## สัดส่วน null ต่อฟีเจอร์ (lookback ตอนต้น series)

Null ที่ส่วนหัวของ series เป็นเรื่องปกติ (ยังมี lookback ไม่พอ) — เราต้องการให้ส่วนกลาง/ท้าย
ไม่มี null ผิดปกติ

In [ ]:
if panel is not None:
    null_frac = panel.null_count() / panel.height
    print(null_frac)

## การกระจายของฟีเจอร์ก่อน/หลัง rollover

เปรียบเทียบ distribution ของฟีเจอร์หลัก (เช่น `atr_ratio`, `ema_slope_20`, `dist_from_vwap`)
ในหน้าต่างก่อน/หลังสัปดาห์ rollover เพื่อยืนยันว่าฟีเจอร์ทำงานบน continuous series ที่ปรับฐานแล้ว
โดยไม่มี jump ที่ขอบสัญญา

In [ ]:
if panel is not None:
    summary = panel.select(
        [pl.col(c).describe() for c in ("atr_ratio", "ema_slope_20", "dist_from_vwap")]
        if all(c in panel.columns for c in ("atr_ratio", "ema_slope_20", "dist_from_vwap"))
        else []
    )
    print(panel.select(["atr_ratio", "ema_slope_20", "dist_from_vwap"]).describe())